# Calculate JPEG compression indicators for multiple images and save them to CSV


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from jpeg_compression_detection import detect_jpeg_compression
from jpeg_compression_detection.paths import GENERATED_DATA_DIR, display_path

image_files_dir = GENERATED_DATA_DIR / "multiple-compression-same-quality"
csv_results_file = GENERATED_DATA_DIR / "compression_analysis_results.csv"

image_files = (
    sorted(
        path
        for path in image_files_dir.iterdir()
        if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".ppm"}
    )
    if image_files_dir.is_dir()
    else []
)

results = []
if not image_files:
    print(f"No repeated-compression images found in {display_path(image_files_dir)}")
else:
    for image_path in image_files:
        print(f"\nAnalyzing: {display_path(image_path)}")
        try:
            compression_result = detect_jpeg_compression(image_path)
            if not isinstance(compression_result, dict):
                print(f"Unable to analyze {image_path.name}: {compression_result}")
                continue
            results.append({"filename": image_path.name, **compression_result})
        except Exception as exc:
            print(f"Error while processing {display_path(image_path)}: {exc}")

    print(f"\nProcessed {len(results)} images successfully")

df = pd.DataFrame(results)
if not df.empty:
    csv_results_file.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(csv_results_file, index=False)


In [ ]:
df.head()

# Visualizations


In [ ]:
csv_file_path = csv_results_file

if csv_file_path.is_file():
    df_file = pd.read_csv(csv_file_path)
    print(f"DataFrame shape: {df_file.shape}")
    display(df_file.head())
else:
    df_file = pd.DataFrame()
    print(f"Visualization skipped; result file is missing: {display_path(csv_file_path)}")


In [ ]:
if df_file.empty:
    print("Plots skipped because no result table is available.")
else:
    results_df = df_file.copy()

    def get_compression_count(filename):
        """Infer the number of JPEG encodings from the generated filename."""
        q_count = filename.count(".q")
        if filename.endswith((".png", ".ppm")):
            return 0
        if q_count == 0:
            return 1
        return q_count + 1

    results_df["compression_count"] = results_df["filename"].apply(get_compression_count)
    fig, axes = plt.subplots(3, 2, figsize=(15, 12))
    colors = ["blue", "green", "orange", "red"]
    compression_levels = sorted(results_df["compression_count"].unique())

    blockiness_data = [results_df.loc[results_df["compression_count"] == level, "blockiness_max"].values for level in compression_levels]
    axes[0, 0].hist(blockiness_data, bins=30, label=[f"{level} compressions" for level in compression_levels], color=colors[: len(compression_levels)], edgecolor="black", stacked=True)
    axes[0, 0].set_xlabel("Maximum blockiness ratio")
    axes[0, 0].set_ylabel("Number of images")
    axes[0, 0].set_title("Maximum blockiness distribution")
    axes[0, 0].axvline(1.1, color="black", linestyle="--", linewidth=2, label="JPEG threshold (1.1)")
    axes[0, 0].legend()

    compression_groups = results_df.groupby("compression_count")["blockiness_max"].apply(list)
    boxplot = axes[0, 1].boxplot(compression_groups.values, labels=compression_groups.index, patch_artist=True)
    for patch, color in zip(boxplot["boxes"], colors[: len(compression_groups)]):
        patch.set_facecolor(color)
        patch.set_alpha(0.5)
    for median in boxplot["medians"]:
        median.set_color("black")
    axes[0, 1].set_xlabel("Number of compressions")
    axes[0, 1].set_ylabel("Maximum blockiness ratio")
    axes[0, 1].set_title("Blockiness vs number of compressions")

    quality_data = [results_df.loc[results_df["compression_count"] == level, "estimated_quality"].values for level in compression_levels]
    axes[1, 0].hist(quality_data, bins=10, label=[f"{level} compressions" for level in compression_levels], color=colors[: len(compression_levels)], edgecolor="black", stacked=True)
    axes[1, 0].set_xlabel("Estimated JPEG quality (%)")
    axes[1, 0].set_ylabel("Number of images")
    axes[1, 0].set_title("Estimated JPEG-quality distribution")
    axes[1, 0].legend()

    first_scatter = axes[1, 1].scatter(results_df["blockiness_max"], results_df["psnr"], c=results_df["compression_count"], cmap="viridis", alpha=0.6, s=20)
    axes[1, 1].set_xlabel("Maximum blockiness ratio")
    axes[1, 1].set_ylabel("PSNR (dB)")
    axes[1, 1].set_title("PSNR vs blockiness")
    plt.colorbar(first_scatter, ax=axes[1, 1], label="Number of compressions")

    energy_data = [results_df.loc[results_df["compression_count"] == level, "energy_mean"].values for level in compression_levels]
    axes[2, 0].hist(energy_data, bins=30, label=[f"{level} compressions" for level in compression_levels], color=colors[: len(compression_levels)], edgecolor="black", stacked=True)
    axes[2, 0].set_xlabel("Mean DCT energy")
    axes[2, 0].set_ylabel("Number of images")
    axes[2, 0].set_title("DCT-energy distribution")
    axes[2, 0].legend()

    second_scatter = axes[2, 1].scatter(results_df["energy_mean"], results_df["energy_std"], c=results_df["compression_count"], cmap="plasma", alpha=0.6, s=20)
    axes[2, 1].set_xlabel("Mean DCT energy")
    axes[2, 1].set_ylabel("DCT-energy standard deviation")
    axes[2, 1].set_title("Mean vs standard deviation of DCT energy")
    plt.colorbar(second_scatter, ax=axes[2, 1], label="Number of compressions")

    plt.tight_layout()
    plt.show()
